# Phase 5 — Figures

Generates Figures 2–7 of *Automated Segmentation and Morphometric Assessment of Neuronal Somata in the Rat Trigeminal Ganglion Using a Fine-Tuned Cellpose-SAM Model* from the result files of Phases 1–3. No GPU is required.

| Figure | Content | Source |
|--------|---------|--------|
| 2 | AP@50/75/90 vs. training epochs (mean ± SD over the 5 test images) | Phase 1, sub-phase 1A |
| 3A / 3B | AP vs. learning rate / weight decay (100 epochs) | Phase 1, sub-phases 1B and 1C |
| 4 | AP heatmaps of the learning rate × weight decay grid | Phase 1, sub-phase 1D |
| 5 | AP across folds, without vs. with augmentation (default inference) | Phase 2 |
| 6 | Validation grid search: mean AP@50 over folds | Phase 3 |
| 7 | AP across folds after inference optimization | Phase 3 |

Precision, Recall and F1 versions of each figure are saved as supplementary panels (`supp_*`). Figures are written as PNG (600 dpi) and TIFF (300 dpi).

**Notebook series** (run in this order; each notebook reads the outputs of the previous ones):

| # | Notebook | Manuscript |
|---|----------|------------|
| 1 | `Phase1_training_hyperparameter_exploration.ipynb` | Methods 2.4.1 · Results 3.1 |
| 2 | `Phase2_generalization_data_augmentation.ipynb` | Methods 2.4.2 · Results 3.2 |
| 3 | `Phase3_inference_parameter_optimization.ipynb` | Methods 2.4.3 · Results 3.3 |
| 4 | `Phase4_morphology_measurements.ipynb` | Methods 2.5 · Results 3.4 · Table 1 · Figures 8–9 |
| 5 | `Phase5_figures.ipynb` | Figures 2–7 (no GPU needed) |

## Required files

| Path variable | Content | Provided file |
|---------------|---------|---------------|
| `PHASE1_DIR` | `phase1_all_results_consolidated.csv` and `<sub-phase>/results_final.json` for `1A_epochs`, `1B_learning_rate`, `1C_weight_decay`, `1D_lr_wd_grid` | `results/phase1/` |
| `PHASE2_DIR` | `phase2_all_results.csv` | `results/phase2/` |
| `PHASE3_DIR` | `phase3_test_evaluation.json`, `phase3_validation_gridsearch.json` | `results/phase3/` |

## Outputs

`FIGURES_DIR/PNG/*.png` and `FIGURES_DIR/TIFF/*.tiff`, with `FIGURES_DIR = PROJECT_DIR/figures`.

**Expected folder layout** (files are distributed separately from the notebooks; set `PROJECT_DIR` in the *Paths* cell to the folder that contains them):

```
PROJECT_DIR/
├── dataset/
│   ├── train/        21 images <id>.tif + Cellpose annotations <id>_seg.npy
│   ├── test/          5 images (1, 7, 10, 15, 17) + <id>_seg.npy
│   └── validation/    5 images (3.2.2, Valid1–Valid4) + <id>_seg.npy
├── models/
│   ├── A_no_augmentation/     A_no_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
│   └── B_with_augmentation/   B_with_augmentation_fold_{1..5}_ep100_lr1e-05_wd0.1.pth
├── results/
│   ├── phase1/  phase2/  phase3/  phase4/
└── figures/
```

## 1. Environment

In [ ]:
%matplotlib inline
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 2. Paths

Only this cell needs to be edited. On Google Colab, mount Google Drive first (`from google.colab import drive; drive.mount('/content/drive')`) and point `PROJECT_DIR` to the folder that contains `dataset/`, `models/` and `results/`.

In [ ]:
PROJECT_DIR = Path(".")

PHASE1_DIR = PROJECT_DIR / "results" / "phase1"
PHASE2_DIR = PROJECT_DIR / "results" / "phase2"
PHASE3_DIR = PROJECT_DIR / "results" / "phase3"
FIGURES_DIR = PROJECT_DIR / "figures"

## 3. Style and helpers

In [ ]:
WONG = {
    "blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "vermil": "#D55E00",
    "skyblue": "#56B4E9", "yellow": "#F0E442", "purple": "#CC79A7", "black": "#000000",
}
IOU_COLORS = {"AP@50": WONG["blue"], "AP@75": WONG["orange"], "AP@90": WONG["green"]}
AUG_COLORS = {"Without augmentation": WONG["blue"], "With augmentation": WONG["green"]}

IOU_KEYS = {"AP@50": "AP@0.5", "AP@75": "AP@0.75", "AP@90": "AP@0.9"}
METRICS = {
    "AP": ("AP", "AP", "Average Precision"),
    "Precision": ("Precision", "Precision", "Precision"),
    "Recall": ("Recall", "Recall", "Recall"),
    "F1": ("F1", "F1", "F1-score"),
}
PRIMARY_IOU = "AP@50"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
    "font.size": 15,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 16,
    "axes.labelweight": "normal",
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.45,
    "legend.frameon": False,
    "legend.fontsize": 13,
    "legend.title_fontsize": 13,
    "lines.linewidth": 3.0,
    "lines.markersize": 5,
})


def csv_col(iou_label, metric):
    return f"{IOU_KEYS[iou_label]}_{METRICS[metric][0]}"


def save_figure(fig, name):
    png_dir, tiff_dir = FIGURES_DIR / "PNG", FIGURES_DIR / "TIFF"
    png_dir.mkdir(parents=True, exist_ok=True)
    tiff_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(png_dir / f"{name}.png", dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(tiff_dir / f"{name}.tiff", dpi=300, bbox_inches="tight", facecolor="white")
    print(f"saved: {name}.png / {name}.tiff")

## 4. Data loading

In [ ]:
PHASE1 = pd.read_csv(PHASE1_DIR / "phase1_all_results_consolidated.csv")
PHASE1.columns = PHASE1.columns.str.strip()
print("Phase 1:", dict(PHASE1["subphase"].value_counts()))

PER_IMAGE = {}
for subphase in ("1A_epochs", "1B_learning_rate", "1C_weight_decay", "1D_lr_wd_grid"):
    with open(PHASE1_DIR / subphase / "results_final.json") as f:
        for entry in json.load(f):
            PER_IMAGE[entry["model_name"]] = entry["per_image_metrics"]
missing = sorted(set(PHASE1["model_name"]) - set(PER_IMAGE))
print(f"Per-image records: {len(PER_IMAGE)} models | missing: {missing}")

PHASE2 = pd.read_csv(PHASE2_DIR / "phase2_all_results.csv")
PHASE2.columns = PHASE2.columns.str.strip()
PHASE2_META = dict(epochs=100, lr="1e-05", wd=0.1)
print(f"Phase 2: {len(PHASE2)} models")


def load_phase3(path):
    rows = []
    for e in json.load(open(path)):
        row = {"fold": e["fold"], "condition": e["condition"],
               "use_augmentation": bool(e["use_augmentation"]), "model_name": e["model_name"]}
        for iou, jk in IOU_KEYS.items():
            for metric in METRICS:
                row[csv_col(iou, metric)] = e["aggregate_metrics"][jk][METRICS[metric][1]]
        row.update({k: e["inference_parameters"].get(k)
                    for k in ("flow_threshold", "cellprob_threshold", "diameter", "min_size")})
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["condition", "fold"]).reset_index(drop=True)


PHASE3 = load_phase3(PHASE3_DIR / "phase3_test_evaluation.json")
PHASE3_META = dict(epochs=100, lr="1e-5", wd=0.1)
PHASE3_GRID = json.load(open(PHASE3_DIR / "phase3_validation_gridsearch.json"))
print(f"Phase 3: {len(PHASE3)} models")
PHASE3[["condition", "fold", "flow_threshold", "cellprob_threshold", "diameter", "min_size"]]

## 5. Aggregation helpers

Line values are the aggregate AP of each model (mean over the test images); shaded bands are ±1 SD over the individual test images (`ddof = 1`).

In [ ]:
def per_image_sd(model_name, iou_label, metric):
    rec = PER_IMAGE.get(model_name)
    if rec is None:
        return np.nan
    arr = np.asarray(rec[IOU_KEYS[iou_label]][METRICS[metric][1]], dtype=float)
    return float(arr.std(ddof=1)) if arr.size > 1 else 0.0


def attach_stats(df, group_col, metric):
    rows = [{group_col: r[group_col], "iou": iou, "value": r[csv_col(iou, metric)],
             "sd": per_image_sd(r["model_name"], iou, metric)}
            for _, r in df.iterrows() for iou in IOU_KEYS]
    return pd.DataFrame(rows).sort_values([group_col, "iou"]).reset_index(drop=True)


def line_plot(df, group_col, metric, *, xlabel, title, fname, log_x=False, legend_loc="best", alpha=0.18):
    tidy = attach_stats(df, group_col, metric)
    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    for iou in IOU_KEYS:
        sub = tidy[tidy.iou == iou].sort_values(group_col)
        x, y, sd = sub[group_col].values, sub["value"].values, np.nan_to_num(sub["sd"].values)
        ax.plot(x, y, marker="o", ms=5, lw=2, color=IOU_COLORS[iou], label=iou, zorder=3)
        ax.fill_between(x, y - sd, y + sd, color=IOU_COLORS[iou], alpha=alpha, lw=0, zorder=1)
    if log_x:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(METRICS[metric][2])
    ax.set_title(title)
    ax.legend(title="IoU threshold", loc=legend_loc)
    fig.tight_layout()
    save_figure(fig, fname)
    plt.show()
    return tidy

## 6. Figure 2 — Number of training epochs

In [ ]:
epoch_df = PHASE1[PHASE1.subphase == "1A_epochs"].copy()

tidy = line_plot(epoch_df, "n_epochs", "AP", xlabel="Training epochs",
                 title="Segmentation performance vs. training epochs", fname="fig2_epochs_AP")
for metric in ("Precision", "Recall", "F1"):
    line_plot(epoch_df, "n_epochs", metric, xlabel="Training epochs",
              title=f"{METRICS[metric][2]} vs. training epochs", fname=f"supp_fig2_epochs_{metric}")

In [ ]:
ap50 = tidy[tidy.iou == PRIMARY_IOU].set_index("n_epochs")
KNEE = 100
peak = int(ap50["value"].idxmax())
se_peak = ap50.loc[peak, "sd"] / np.sqrt(5)

print(f"AP@50: {ap50['value'].iloc[0]:.3f} (1 epoch) -> {ap50.loc[KNEE, 'value']:.3f} ({KNEE} epochs)")
print(f"Range from {KNEE} to 500 epochs: {ap50.loc[KNEE:, 'value'].max() - ap50.loc[KNEE:, 'value'].min():.3f}")
print(f"Peak: {peak} epochs, AP@50 = {ap50.loc[peak, 'value']:.3f} ± {ap50.loc[peak, 'sd']:.3f}")
print(f"One-SE threshold: {ap50.loc[peak, 'value'] - se_peak:.3f}")
print(f"{KNEE} epochs: AP@50 = {ap50.loc[KNEE, 'value']:.3f} ± {ap50.loc[KNEE, 'sd']:.3f}")
times = epoch_df.set_index("n_epochs")["training_time_s"]
print(f"Training time: {times[KNEE]:.1f} s ({KNEE} epochs) vs. {times[peak]:.1f} s ({peak} epochs)")

## 7. Figure 3 — Learning rate (A) and weight decay (B)

In [ ]:
lr_df = PHASE1[PHASE1.subphase == "1B_learning_rate"]
wd_df = PHASE1[PHASE1.subphase == "1C_weight_decay"]

line_plot(lr_df, "learning_rate", "AP", xlabel="Learning rate", title="Learning-rate effect on AP metrics",
          fname="fig3a_learning_rate_AP", log_x=True, legend_loc="lower left", alpha=0.15)
line_plot(wd_df, "weight_decay", "AP", xlabel="Weight decay", title="Weight-decay effect on AP metrics",
          fname="fig3b_weight_decay_AP", log_x=True, legend_loc="lower left", alpha=0.15)

for metric in ("Precision", "Recall", "F1"):
    line_plot(lr_df, "learning_rate", metric, xlabel="Learning rate",
              title=f"Learning-rate effect on {METRICS[metric][2]}",
              fname=f"supp_fig3a_learning_rate_{metric}", log_x=True, legend_loc="lower left", alpha=0.15)
    line_plot(wd_df, "weight_decay", metric, xlabel="Weight decay",
              title=f"Weight-decay effect on {METRICS[metric][2]}",
              fname=f"supp_fig3b_weight_decay_{metric}", log_x=True, legend_loc="lower left", alpha=0.15)

## 8. Figure 4 — Learning rate × weight decay grid (100 epochs)

In [ ]:
def heatmap_grid(df, metric, fname):
    grid_df = df[df.subphase == "1D_lr_wd_grid"].copy()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    all_vals = np.concatenate([grid_df[csv_col(iou, metric)].values for iou in IOU_KEYS])
    vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
    im = None
    for ax, iou in zip(axes, IOU_KEYS):
        pivot = grid_df.pivot(index="learning_rate", columns="weight_decay",
                              values=csv_col(iou, metric)).sort_index(ascending=False)
        im = ax.imshow(pivot.values, cmap="YlGnBu", vmin=vmin, vmax=vmax)
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_xticklabels([f"{x:g}" for x in pivot.columns])
        ax.set_yticklabels([f"{x:g}" for x in pivot.index])
        ax.set_title(iou)
        ax.set_xlabel("Weight decay")
        if ax is axes[0]:
            ax.set_ylabel("Learning rate")
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9,
                        color="white" if val > (vmin + vmax) / 2 else "black")
    fig.suptitle(f"{METRICS[metric][2]} — lr x wd grid (100 epochs)", fontsize=14, fontweight="bold")
    plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label=METRICS[metric][2])
    save_figure(fig, fname)
    plt.show()


heatmap_grid(PHASE1, "AP", "fig4_lr_wd_grid_AP")
for metric in ("Precision", "Recall", "F1"):
    heatmap_grid(PHASE1, metric, f"supp_fig4_lr_wd_grid_{metric}")

grid = PHASE1[PHASE1.subphase == "1D_lr_wd_grid"]
best = grid.loc[grid["AP@0.5_AP"].idxmax()]
print(f"Best cell: lr = {best['learning_rate']:g}, wd = {best['weight_decay']:g} | "
      f"AP@50 = {best['AP@0.5_AP']:.3f}, AP@75 = {best['AP@0.75_AP']:.3f}, AP@90 = {best['AP@0.9_AP']:.3f}")
print(f"AP@50 range: {grid['AP@0.5_AP'].min():.3f} to {grid['AP@0.5_AP'].max():.3f}")

## 9. Figure 5 — Cross-validation, without vs. with augmentation (default inference)

In [ ]:
rng = np.random.default_rng(42)


def fold_boxplots(df, meta, metric, config_label, fname):
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.8), sharey=True)
    conds = [("Without augmentation", False), ("With augmentation", True)]
    for ax, iou in zip(axes, IOU_KEYS):
        col = csv_col(iou, metric)
        data = [df[df.use_augmentation == flag][col].dropna().values for _, flag in conds]
        colors = [AUG_COLORS[name] for name, _ in conds]

        bp = ax.boxplot(data, widths=0.55, patch_artist=True,
                        medianprops=dict(color="black", lw=2),
                        whiskerprops=dict(color="0.3"), capprops=dict(color="0.3"),
                        flierprops=dict(marker="none"))
        for patch, c in zip(bp["boxes"], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.45)
            patch.set_edgecolor(c)

        for pos, (vals, c) in enumerate(zip(data, colors), start=1):
            ax.scatter(rng.normal(pos, 0.05, size=len(vals)), vals, s=42, color=c,
                       edgecolor="black", lw=0.5, zorder=3)
            ax.text(pos, vals.max() + 0.015, f"$\\mu$={vals.mean():.3f}\n$\\sigma$={vals.std(ddof=1):.3f}",
                    ha="center", va="bottom", fontsize=8, fontweight="bold")

        ax.set_xticks([1, 2])
        ax.set_xticklabels(["Without\naug.", "With\naug."])
        ax.set_title(iou)
        if ax is axes[0]:
            ax.set_ylabel(METRICS[metric][2])
        ax.margins(y=0.18)

    fig.suptitle(f"{METRICS[metric][2]} across folds — {config_label}  "
                 f"(ep={meta['epochs']}, lr={meta['lr']}, wd={meta['wd']:g})", fontsize=13, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_figure(fig, fname)
    plt.show()


fold_boxplots(PHASE2, PHASE2_META, "AP", "100 epochs", "fig5_phase2_folds_AP")
for metric in ("Precision", "Recall", "F1"):
    fold_boxplots(PHASE2, PHASE2_META, metric, "100 epochs", f"supp_fig5_phase2_folds_{metric}")

for flag, label in ((False, "Without augmentation"), (True, "With augmentation")):
    sub = PHASE2[PHASE2.use_augmentation == flag]
    print(label + ": " + " | ".join(
        f"{c} = {sub[c].mean():.3f} ± {sub[c].std():.3f}"
        for c in ("AP@0.5_AP", "AP@0.75_AP", "AP@0.9_AP", "AP@0.5_Precision", "AP@0.5_Recall", "AP@0.5_F1")))

## 10. Figure 6 — Validation grid search of the inference parameters

In [ ]:
def phase3_grid_heatmap(data, fname):
    gdf = pd.DataFrame([{"condition": e["condition"], "flow": c["flow_threshold"],
                         "cellprob": c["cellprob_threshold"], "AP50": c["aggregate"]["AP@0.5"]["AP"]}
                        for e in data for c in e["all_combos"]])
    conds = ["A_no_augmentation", "B_with_augmentation"]
    titles = {"A_no_augmentation": "Without augmentation", "B_with_augmentation": "With augmentation"}
    vmin, vmax = gdf["AP50"].min(), gdf["AP50"].max()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    im = None
    for ax, cond in zip(axes, conds):
        piv = (gdf[gdf.condition == cond]
               .pivot_table(index="cellprob", columns="flow", values="AP50", aggfunc="mean")
               .sort_index(ascending=False))
        im = ax.imshow(piv.values, cmap="YlGnBu", vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(np.arange(len(piv.columns)))
        ax.set_xticklabels([f"{x:g}" for x in piv.columns])
        ax.set_yticks(np.arange(len(piv.index)))
        ax.set_yticklabels([f"{x:g}" for x in piv.index])
        ax.set_title(titles[cond])
        ax.set_xlabel("flow_threshold")
        if ax is axes[0]:
            ax.set_ylabel("cellprob_threshold")
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                v = piv.values[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if v > (vmin + vmax) / 2 else "black")
    fig.suptitle("Phase 3 - validation grid search: mean AP@50 over folds", fontsize=14, fontweight="bold")
    plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8, label="AP@50")
    save_figure(fig, fname)
    plt.show()


phase3_grid_heatmap(PHASE3_GRID, "fig6_phase3_validation_grid")

val = pd.DataFrame([{"condition": e["condition"], "fold": e["fold"],
                     "flow": e["best_flow_threshold"], "cellprob": e["best_cellprob_threshold"],
                     **{k: e["best_validation_metrics"][IOU_KEYS[k]]["AP"] for k in IOU_KEYS}}
                    for e in PHASE3_GRID])
print(val.to_string(index=False))
print(val.groupby("condition")[list(IOU_KEYS)].agg(["mean", "std"]).round(3))

## 11. Figure 7 — Cross-validation after inference optimization

In [ ]:
fold_boxplots(PHASE3, PHASE3_META, "AP", "Phase 3 - optimized inference", "fig7_phase3_folds_AP")
for metric in ("Precision", "Recall", "F1"):
    fold_boxplots(PHASE3, PHASE3_META, metric, "Phase 3 - optimized inference", f"supp_fig7_phase3_folds_{metric}")

for flag, label in ((False, "Without augmentation"), (True, "With augmentation")):
    sub = PHASE3[PHASE3.use_augmentation == flag]
    print(label + ": " + " | ".join(
        f"{c} = {sub[c].mean():.3f} ± {sub[c].std():.3f}"
        for c in ("AP@0.5_AP", "AP@0.75_AP", "AP@0.9_AP", "AP@0.5_Precision", "AP@0.5_Recall", "AP@0.5_F1")))